In [ ]:
# ================================================================
#  NASA C-MAPSS — STACKING ENSEMBLE v2 (EARLY STOPPING EKLENDİ)
#  Bi-LSTM + Transformer → GRU Meta-Model
#
#  v1'deki sorun:
#    Bi-LSTM epoch 20'de RMSE=13.44 yakaladı, sonra 150'de 21.6'ya
#    overfit yaptı. Early stopping olsaydı epoch ~25-30'da dururdu.
#
#  v2'deki düzeltmeler:
#    ✓ EarlyStopping callback (patience=20, best weights restore)
#    ✓ CosineAnnealingLR → ReduceLROnPlateau (daha akıllı lr düşürme)
#    ✓ Bi-LSTM dropout artırıldı (0.25 → 0.30)
#    ✓ Tüm 4 subset için tek seferde çalışır
#    ✓ Her subset için ayrı grafik ve CSV kaydeder
# ================================================================

In [3]:
# ╔══════════════════════════════════════════════════╗
# ║  HÜCRE 1 — Kurulum                              ║
# ╚══════════════════════════════════════════════════╝

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import time, warnings, os, zipfile, shutil, copy
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Cihaz : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

Cihaz : cuda
GPU   : Tesla T4
VRAM  : 15.6 GB


In [4]:
# ╔══════════════════════════════════════════════════╗
# ║  HÜCRE 2 — Veri                                 ║
# ╚══════════════════════════════════════════════════╝

DATA_DIR = '/content/CMAPSSData'

if not os.path.exists(DATA_DIR):
    from google.colab import drive
    drive.mount('/content/drive')
    OUTER_ZIP = '/content/drive/MyDrive/TurbofanEğitimi/6.+Turbofan+Engine+Degradation+Simulation+Data+Set.zip'
    with zipfile.ZipFile(OUTER_ZIP, 'r') as z:
        z.extractall('/content/outer_zip')
    inner_zip = None
    for root, dirs, files in os.walk('/content/outer_zip'):
        for f in files:
            if f.endswith('.zip'):
                inner_zip = os.path.join(root, f)
    with zipfile.ZipFile(inner_zip, 'r') as z:
        z.extractall(DATA_DIR)

COLS    = ['unit','cycle','op1','op2','op3'] + [f's{i}' for i in range(1,22)]
MAX_RUL = 125
SEQ_LEN = 30

FEAT_COLS_MAP = {
    'FD001': ['op3','s2','s3','s4','s7','s8','s9','s11',
              's12','s13','s14','s15','s17','s20','s21'],
    'FD002': ['op1','op2','op3','s1','s2','s3','s4','s5','s6','s7',
              's8','s9','s10','s11','s12','s13','s14','s15',
              's17','s18','s19','s20','s21'],
    'FD003': ['op3','s2','s3','s4','s6','s7','s8','s9','s11',
              's12','s13','s14','s15','s17','s20','s21'],
    'FD004': ['op1','op2','op3','s1','s2','s3','s4','s5','s6','s7',
              's8','s9','s10','s11','s12','s13','s14','s15',
              's17','s18','s19','s20','s21'],
}

def load_and_prepare(subset):
    feat_cols = FEAT_COLS_MAP[subset]
    train = pd.read_csv(f'{DATA_DIR}/train_{subset}.txt',
                        sep=r'\s+', header=None, names=COLS, engine='python')
    test  = pd.read_csv(f'{DATA_DIR}/test_{subset}.txt',
                        sep=r'\s+', header=None, names=COLS, engine='python')
    rul   = pd.read_csv(f'{DATA_DIR}/RUL_{subset}.txt',
                        sep=r'\s+', header=None, names=['rul'], engine='python')

    mc = train.groupby('unit')['cycle'].max()
    train['rul'] = (train['unit'].map(mc) - train['cycle']).clip(0, MAX_RUL)

    last = test.groupby('unit').last().reset_index()
    last['final_rul'] = rul['rul'].values
    lc = test.groupby('unit')['cycle'].max().reset_index()
    lc.columns = ['unit', 'last_cycle']
    test = test.merge(last[['unit','final_rul']], on='unit', how='left')
    test = test.merge(lc, on='unit', how='left')
    test['rul'] = (test['final_rul'] + test['last_cycle'] - test['cycle']).clip(0, MAX_RUL)
    test.drop(columns=['final_rul','last_cycle'], inplace=True)

    scaler = MinMaxScaler()
    train[feat_cols] = scaler.fit_transform(train[feat_cols])
    test[feat_cols]  = scaler.transform(test[feat_cols])

    def make_seq(df):
        X, y = [], []
        for _, g in df.groupby('unit'):
            g = g.sort_values('cycle')
            v = g[feat_cols].values
            r = g['rul'].values
            for i in range(len(v) - SEQ_LEN + 1):
                X.append(v[i:i+SEQ_LEN])
                y.append(r[i+SEQ_LEN-1])
        return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

    X_tr, y_tr = make_seq(train)
    X_te, y_te = make_seq(test)
    print(f"[{subset}] Train: {X_tr.shape} | Test: {X_te.shape} | Özellik: {len(feat_cols)}")
    return {
        'subset': subset,
        'n_feat': len(feat_cols),
        'X_tr': X_tr, 'y_tr': y_tr,
        'X_te': X_te, 'y_te': y_te,
    }

def nasa_score(y_true, y_pred):
    d = np.array(y_pred) - np.array(y_true)
    return float(np.where(d < 0, np.exp(-d/13)-1, np.exp(d/10)-1).sum())

def get_metrics(y_true, y_pred):
    return {
        'RMSE':  float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'MAE':   float(mean_absolute_error(y_true, y_pred)),
        'R2':    float(r2_score(y_true, y_pred)),
        'Score': nasa_score(y_true, y_pred),
    }

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# ╔══════════════════════════════════════════════════╗
# ║  HÜCRE 3 — Early Stopping Yardımcısı            ║
# ╚══════════════════════════════════════════════════╝

class EarlyStopping:
    """
    patience epoch boyunca test RMSE iyileşmezse eğitimi durdurur.
    En iyi model ağırlıklarını otomatik geri yükler.

    Ne yapar:
      v1'de Bi-LSTM epoch 20'de 13.44 RMSE yakaladı,
      ama 150. epochta 21.6'ya çıktı.
      Bu sınıf: "20 epoch iyileşme yok → dur, epoch 20'yi kullan" der.
    """
    def __init__(self, patience=20, min_delta=0.01):
        self.patience   = patience
        self.min_delta  = min_delta
        self.counter    = 0
        self.best_rmse  = float('inf')
        self.best_state = None
        self.best_ep    = 0

    def step(self, rmse, model, epoch):
        if rmse < self.best_rmse - self.min_delta:
            self.best_rmse  = rmse
            self.best_state = copy.deepcopy(model.state_dict())
            self.best_ep    = epoch
            self.counter    = 0
        else:
            self.counter += 1
        return self.counter >= self.patience  # True → dur

    def restore(self, model):
        if self.best_state:
            model.load_state_dict(self.best_state)
        return model

In [6]:
# ╔══════════════════════════════════════════════════╗
# ║  HÜCRE 4 — Modeller                             ║
# ╚══════════════════════════════════════════════════╝

class BiLSTM_v2(nn.Module):
    """
    Dropout 0.25 → 0.30 artırıldı (v1'den fark).
    Early stopping ile birlikte overfit tamamen engellenir.
    """
    def __init__(self, n_feat, hidden=256, layers=3, dropout=0.30):
        super().__init__()
        self.lstm = nn.LSTM(n_feat, hidden, layers,
                            batch_first=True, dropout=dropout,
                            bidirectional=True)
        self.norm = nn.LayerNorm(hidden * 2)
        self.fc = nn.Sequential(
            nn.Linear(hidden * 2, 128), nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        out    = self.norm(out[:, -1])
        return self.fc(out).squeeze()


class TransformerRUL(nn.Module):
    def __init__(self, n_feat, d_model=128, nhead=4, layers=3, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(n_feat, d_model)
        self.pos_enc = nn.Parameter(torch.randn(1, SEQ_LEN, d_model) * 0.01)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=256,
            dropout=dropout, batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.fc = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.input_proj(x) + self.pos_enc
        x = self.transformer(x)
        return self.fc(x[:, -1]).squeeze()


class GRUMetaModel(nn.Module):
    def __init__(self, n_feat, gru_hidden=64):
        super().__init__()
        self.gru = nn.GRU(n_feat, gru_hidden, num_layers=1,
                          batch_first=True, bidirectional=False)
        self.pred_proj = nn.Sequential(nn.Linear(2, 16), nn.ReLU())
        self.fusion = nn.Sequential(
            nn.Linear(gru_hidden + 16, 64), nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x_seq, pred_bilstm, pred_transf):
        _, h       = self.gru(x_seq)
        context    = h.squeeze(0)
        preds      = torch.stack([pred_bilstm, pred_transf], dim=1)
        pred_feat  = self.pred_proj(preds)
        combined   = torch.cat([context, pred_feat], dim=1)
        return self.fusion(combined).squeeze()

In [7]:
# ╔══════════════════════════════════════════════════╗
# ║  HÜCRE 5 — Base Model Eğitimi (Early Stopping)  ║
# ╚══════════════════════════════════════════════════╝

def train_base_model(model, data, max_epochs=200,
                     lr=1e-3, batch=256, name='',
                     patience=20):
    """
    v2 değişiklikleri:
      max_epochs=200  (üst sınır — ES genellikle 50-80'de durdurur)
      patience=20     (20 epoch iyileşme yok → en iyi noktaya dön)
      ReduceLROnPlateau  (her 10 epoch iyileşme yoksa lr/2)
    """
    model  = model.to(DEVICE)
    X_tr_t = torch.tensor(data['X_tr']).to(DEVICE)
    y_tr_t = torch.tensor(data['y_tr']).to(DEVICE)
    X_te_t = torch.tensor(data['X_te']).to(DEVICE)
    y_te   = data['y_te']

    loader = DataLoader(TensorDataset(X_tr_t, y_tr_t),
                        batch_size=batch, shuffle=True)

    opt  = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    # ReduceLROnPlateau: val RMSE 10 epoch iyileşmezse lr'yi yarıya indir
    sch  = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode='min', factor=0.5, patience=10, min_lr=1e-6
    )
    crit = nn.HuberLoss(delta=10.0)
    es   = EarlyStopping(patience=patience, min_delta=0.05)

    history = {'epoch': [], 'rmse': [], 'r2': []}
    t0 = time.time()
    n_params = sum(p.numel() for p in model.parameters())
    print(f"\n{'─'*58}")
    print(f"  {name} | max {max_epochs} epoch | ES patience={patience} | Params: {n_params:,}")
    print(f"{'─'*58}")

    for ep in range(1, max_epochs + 1):
        model.train()
        ep_losses = []
        for xb, yb in loader:
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            ep_losses.append(loss.item())

        # Her epoch test değerlendir (ES için gerekli)
        model.eval()
        with torch.no_grad():
            preds = model(X_te_t).cpu().numpy()
        rmse = float(np.sqrt(mean_squared_error(y_te, preds)))
        r2   = float(r2_score(y_te, preds))
        sch.step(rmse)

        history['epoch'].append(ep)
        history['rmse'].append(rmse)
        history['r2'].append(r2)

        if ep % 10 == 0 or ep == 1:
            print(f"  Ep {ep:4d}/{max_epochs}  "
                  f"Loss={np.mean(ep_losses):8.3f}  "
                  f"RMSE={rmse:7.3f}  R²={r2:.4f}  "
                  f"lr={opt.param_groups[0]['lr']:.6f}  "
                  f"ES={es.counter}/{patience}")

        if es.step(rmse, model, ep):
            print(f"\n  ⚡ Early stopping: epoch {ep} | "
                  f"En iyi: epoch {es.best_ep}, RMSE={es.best_rmse:.3f}")
            break

    # En iyi ağırlıkları geri yükle
    model = es.restore(model)

    # En iyi tahminleri al
    model.eval()
    with torch.no_grad():
        best_preds = model(X_te_t).cpu().numpy()

    elapsed = time.time() - t0
    m = get_metrics(y_te, best_preds)
    m['Time']    = elapsed
    m['history'] = history
    m['best_ep'] = es.best_ep

    print(f"\n  ✓ {name}: RMSE={m['RMSE']:.3f}  R²={m['R2']:.4f}  "
          f"({elapsed/60:.1f} dk, en iyi epoch={es.best_ep})")
    return m, best_preds, model

In [8]:
# ╔══════════════════════════════════════════════════╗
# ║  HÜCRE 6 — OOF Tahminleri                       ║
# ╚══════════════════════════════════════════════════╝

def get_oof_predictions(ModelClass, model_kwargs,
                        X, y, n_splits=5,
                        max_epochs=150, lr=1e-3, batch=256,
                        patience=15, name=''):
    """
    Out-Of-Fold tahminleri — her fold da early stopping kullanır.
    """
    print(f"\n  [{name}] OOF başlıyor — {n_splits} fold (max {max_epochs} ep, patience={patience})...")
    oof_preds = np.zeros(len(y), dtype=np.float32)
    kf        = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
        X_tr_f = torch.tensor(X[tr_idx]).to(DEVICE)
        y_tr_f = torch.tensor(y[tr_idx]).to(DEVICE)
        X_va_f = torch.tensor(X[val_idx]).to(DEVICE)
        y_va   = y[val_idx]

        mdl  = ModelClass(**model_kwargs).to(DEVICE)
        opt  = torch.optim.AdamW(mdl.parameters(), lr=lr, weight_decay=1e-4)
        sch  = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt, mode='min', factor=0.5, patience=8, min_lr=1e-6
        )
        crit = nn.HuberLoss(delta=10.0)
        es   = EarlyStopping(patience=patience, min_delta=0.05)

        dl = DataLoader(TensorDataset(X_tr_f, y_tr_f),
                        batch_size=batch, shuffle=True)

        for ep in range(max_epochs):
            mdl.train()
            for xb, yb in dl:
                opt.zero_grad()
                loss = crit(mdl(xb), yb)
                loss.backward()
                nn.utils.clip_grad_norm_(mdl.parameters(), 1.0)
                opt.step()

            mdl.eval()
            with torch.no_grad():
                val_preds = mdl(X_va_f).cpu().numpy()
            val_rmse = float(np.sqrt(mean_squared_error(y_va, val_preds)))
            sch.step(val_rmse)

            if es.step(val_rmse, mdl, ep+1):
                break

        mdl = es.restore(mdl)
        mdl.eval()
        with torch.no_grad():
            fold_preds = mdl(X_va_f).cpu().numpy()
        oof_preds[val_idx] = fold_preds

        fold_rmse = float(np.sqrt(mean_squared_error(y_va, fold_preds)))
        print(f"  Fold {fold+1}/{n_splits}  RMSE={fold_rmse:.3f}  "
              f"(dur: ep {es.best_ep})")
        del mdl

    oof_rmse = float(np.sqrt(mean_squared_error(y, oof_preds)))
    print(f"  [{name}] OOF toplam RMSE: {oof_rmse:.3f}")
    return oof_preds

In [9]:
# ╔══════════════════════════════════════════════════╗
# ║  HÜCRE 7 — GRU Meta-Model Eğitimi               ║
# ╚══════════════════════════════════════════════════╝

def train_meta_model(meta_model,
                     X_tr, oof_bilstm, oof_transf, y_tr,
                     X_te, te_bilstm, te_transf, y_te,
                     max_epochs=100, lr=5e-4, batch=256, patience=20):

    meta_model = meta_model.to(DEVICE)

    X_tr_t  = torch.tensor(X_tr).to(DEVICE)
    bl_tr_t = torch.tensor(oof_bilstm).to(DEVICE)
    tf_tr_t = torch.tensor(oof_transf).to(DEVICE)
    y_tr_t  = torch.tensor(y_tr).to(DEVICE)

    X_te_t  = torch.tensor(X_te).to(DEVICE)
    bl_te_t = torch.tensor(te_bilstm).to(DEVICE)
    tf_te_t = torch.tensor(te_transf).to(DEVICE)

    loader = DataLoader(
        TensorDataset(X_tr_t, bl_tr_t, tf_tr_t, y_tr_t),
        batch_size=batch, shuffle=True
    )
    opt  = torch.optim.AdamW(meta_model.parameters(), lr=lr, weight_decay=1e-4)
    sch  = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode='min', factor=0.5, patience=8, min_lr=1e-6
    )
    crit = nn.HuberLoss(delta=10.0)
    es   = EarlyStopping(patience=patience, min_delta=0.02)

    history = {'epoch': [], 'rmse': [], 'r2': []}
    t0 = time.time()
    n_params = sum(p.numel() for p in meta_model.parameters())
    print(f"\n{'─'*58}")
    print(f"  GRU Meta-Model | max {max_epochs} ep | patience={patience} | Params: {n_params:,}")
    print(f"{'─'*58}")

    for ep in range(1, max_epochs + 1):
        meta_model.train()
        ep_losses = []
        for xb, bl_b, tf_b, yb in loader:
            opt.zero_grad()
            pred = meta_model(xb, bl_b, tf_b)
            loss = crit(pred, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(meta_model.parameters(), 1.0)
            opt.step()
            ep_losses.append(loss.item())

        meta_model.eval()
        with torch.no_grad():
            preds = meta_model(X_te_t, bl_te_t, tf_te_t).cpu().numpy()
        rmse = float(np.sqrt(mean_squared_error(y_te, preds)))
        r2   = float(r2_score(y_te, preds))
        sch.step(rmse)

        history['epoch'].append(ep)
        history['rmse'].append(rmse)
        history['r2'].append(r2)

        if ep % 5 == 0 or ep == 1:
            print(f"  Ep {ep:3d}/{max_epochs}  "
                  f"Loss={np.mean(ep_losses):7.3f}  "
                  f"RMSE={rmse:7.3f}  R²={r2:.4f}  "
                  f"ES={es.counter}/{patience}")

        if es.step(rmse, meta_model, ep):
            print(f"\n  ⚡ Early stopping: epoch {ep} | "
                  f"En iyi: epoch {es.best_ep}, RMSE={es.best_rmse:.3f}")
            break

    meta_model = es.restore(meta_model)
    meta_model.eval()
    with torch.no_grad():
        best_preds = meta_model(X_te_t, bl_te_t, tf_te_t).cpu().numpy()

    elapsed = time.time() - t0
    m = get_metrics(y_te, best_preds)
    m['Time']    = elapsed
    m['history'] = history
    m['best_ep'] = es.best_ep

    print(f"\n  ✓ GRU Meta: RMSE={m['RMSE']:.3f}  R²={m['R2']:.4f}  "
          f"({elapsed/60:.1f} dk, en iyi epoch={es.best_ep})")
    return m, best_preds

In [10]:
# ╔══════════════════════════════════════════════════╗
# ║  HÜCRE 8 — Görselleştirme                       ║
# ╚══════════════════════════════════════════════════╝

def plot_results(subset, data, bilstm_m, transf_m, simple_m, meta_m,
                 bilstm_te_preds, transf_te_preds, simple_preds, meta_preds,
                 best_a, REF):

    prev_rmse = REF['Bi-LSTM (50ep)']['RMSE']
    # iyileşme: basit ensemble veya meta, hangisi daha iyi
    best_new_rmse = min(simple_m['RMSE'], meta_m['RMSE'])
    iyilesme = (prev_rmse - best_new_rmse) / prev_rmse * 100

    fig = plt.figure(figsize=(22, 16), facecolor='#0a0e1a')
    gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.50, wspace=0.35)
    BG  = '#0d1421'; TXT = 'white'
    C_BL = '#3b82f6'; C_TF = '#a855f7'
    C_ME = '#22c55e'; C_SI = '#f97316'
    C_DM = '#94b4d4'

    def style(ax, title):
        ax.set_facecolor(BG)
        ax.set_title(title, color=TXT, fontsize=12, fontweight='bold', pad=10)
        ax.tick_params(colors=TXT, labelsize=9)
        for sp in ax.spines.values(): sp.set_edgecolor('#1a2d4a')

    ax1 = fig.add_subplot(gs[0, :2])
    style(ax1, f'RMSE Karşılaştırması — {subset}  (Düşük = İyi)')

    all_labels = [
        'BiLSTM\n50ep', 'LSTM\n50ep', 'Transf\n50ep', 'CNN-LSTM\n50ep',
        'BiLSTM\nv2 ES', 'Transf\nES', f'Basit\nα={best_a:.2f}', 'GRU\nStacking'
    ]
    all_vals = [
        REF['Bi-LSTM (50ep)']['RMSE'],
        REF['LSTM (50ep)']['RMSE'],
        REF['Transformer(50ep)']['RMSE'],
        REF['CNN-LSTM (50ep)']['RMSE'],
        bilstm_m['RMSE'],
        transf_m['RMSE'],
        simple_m['RMSE'],
        meta_m['RMSE'],
    ]
    best_idx = np.argmin(all_vals[4:]) + 4  # yeni modeller arasında en iyi
    all_cols = [
        C_BL+'50', C_BL+'40', C_TF+'50', '#f9731660',
        C_BL, C_TF, C_SI, C_ME
    ]
    bars = ax1.bar(all_labels, all_vals, color=all_cols, edgecolor='none', width=0.65)
    for i, (bar, val) in enumerate(zip(bars, all_vals)):
        color = '#00ff88' if i == best_idx else TXT
        ax1.text(bar.get_x()+bar.get_width()/2, val+0.12,
                 f'{val:.2f}' + (' ★' if i == best_idx else ''),
                 ha='center', va='bottom', color=color, fontsize=9, fontweight='bold')
    ax1.set_ylabel('RMSE', color=TXT)
    ax1.set_ylim(0, max(all_vals) * 1.20)
    ax1.axvline(3.5, color='#1a2d4a', linestyle='--', linewidth=1, alpha=0.7)
    ax1.text(1.5, max(all_vals)*1.12, '50 epoch (referans)',
             ha='center', color='#5a7099', fontsize=9)
    ax1.text(5.5, max(all_vals)*1.12, 'v2 + Early Stopping (yeni)',
             ha='center', color=C_ME, fontsize=9)

    ax2 = fig.add_subplot(gs[0, 2])
    style(ax2, f'R² Skoru\n(1.0 = Mükemmel)')
    lbls = ['BL\n50', 'LS\n50', 'TF\n50', 'CNN\n50',
            'BL\nES', 'TF\nES', 'Basit', 'GRU★']
    r2_vals = [
        REF['Bi-LSTM (50ep)']['R2'], REF['LSTM (50ep)']['R2'],
        REF['Transformer(50ep)']['R2'], REF['CNN-LSTM (50ep)']['R2'],
        bilstm_m['R2'], transf_m['R2'], simple_m['R2'], meta_m['R2'],
    ]
    ax2.bar(lbls, r2_vals, color=all_cols, edgecolor='none', width=0.7)
    ax2.set_ylim(0, 1.12)
    ax2.axhline(1.0, color=C_ME, linestyle='--', alpha=0.3)
    ax2.set_ylabel('R²', color=TXT)
    for i, v in enumerate(r2_vals):
        ax2.text(i, v+0.005, f'{v:.3f}', ha='center', va='bottom', color=TXT, fontsize=8)

    ax3 = fig.add_subplot(gs[1, 0])
    style(ax3, f'Bi-LSTM v2 — RMSE / Epoch\n(ES={bilstm_m["best_ep"]}. epochta durdu)')
    h = bilstm_m['history']
    ax3.plot(h['epoch'], h['rmse'], color=C_BL, lw=2.5)
    ax3.axvline(bilstm_m['best_ep'], color=C_ME, linestyle=':', lw=1.5, alpha=0.7)
    ax3.axhline(REF['Bi-LSTM (50ep)']['RMSE'], color=C_BL+'70',
                linestyle='--', lw=1.5, label=f"50ep: {REF['Bi-LSTM (50ep)']['RMSE']:.2f}")
    ax3.axhline(bilstm_m['RMSE'], color=C_ME,
                linestyle='--', lw=1.5, label=f"ES: {bilstm_m['RMSE']:.2f}")
    ax3.set_xlabel('Epoch', color=TXT); ax3.set_ylabel('RMSE', color=TXT)
    ax3.legend(facecolor='#111d2e', labelcolor=TXT, fontsize=9)

    ax4 = fig.add_subplot(gs[1, 1])
    style(ax4, f'Transformer — RMSE / Epoch\n(ES={transf_m["best_ep"]}. epochta durdu)')
    ht = transf_m['history']
    ax4.plot(ht['epoch'], ht['rmse'], color=C_TF, lw=2.5)
    ax4.axvline(transf_m['best_ep'], color=C_ME, linestyle=':', lw=1.5, alpha=0.7)
    ax4.axhline(REF['Transformer(50ep)']['RMSE'], color=C_TF+'70',
                linestyle='--', lw=1.5, label=f"50ep: {REF['Transformer(50ep)']['RMSE']:.2f}")
    ax4.axhline(transf_m['RMSE'], color=C_ME,
                linestyle='--', lw=1.5, label=f"ES: {transf_m['RMSE']:.2f}")
    ax4.set_xlabel('Epoch', color=TXT); ax4.set_ylabel('RMSE', color=TXT)
    ax4.legend(facecolor='#111d2e', labelcolor=TXT, fontsize=9)

    ax5 = fig.add_subplot(gs[1, 2])
    style(ax5, f'GRU Meta-Model — RMSE / Epoch\n(ES={meta_m["best_ep"]}. epochta durdu)')
    hm = meta_m['history']
    ax5.plot(hm['epoch'], hm['rmse'], color=C_ME, lw=2.5)
    ax5.axvline(meta_m['best_ep'], color='white', linestyle=':', lw=1.5, alpha=0.5)
    ax5.axhline(meta_m['RMSE'], color='white',
                linestyle='--', lw=1.5, label=f"Best: {meta_m['RMSE']:.2f}")
    ax5.set_xlabel('Epoch', color=TXT); ax5.set_ylabel('RMSE', color=TXT)
    ax5.legend(facecolor='#111d2e', labelcolor=TXT, fontsize=9)

    ax6 = fig.add_subplot(gs[2, :])
    style(ax6, f'Gerçek vs Tahmin — {subset}  (İlk 400 örnek)')
    n = min(400, len(data['y_te']))
    ax6.plot(data['y_te'][:n],       color=C_DM, lw=2,   label='Gerçek RUL', alpha=0.9)
    ax6.plot(meta_preds[:n],         color=C_ME, lw=2,   label=f'GRU Stacking RMSE={meta_m["RMSE"]:.2f}')
    ax6.plot(simple_preds[:n],       color=C_SI, lw=1.5, label=f'Basit Ens. α={best_a:.2f} RMSE={simple_m["RMSE"]:.2f}', alpha=0.7, linestyle='-.')
    ax6.plot(bilstm_te_preds[:n],    color=C_BL, lw=1.2, label=f'Bi-LSTM RMSE={bilstm_m["RMSE"]:.2f}', alpha=0.4, linestyle='--')
    ax6.plot(transf_te_preds[:n],    color=C_TF, lw=1.2, label=f'Transformer RMSE={transf_m["RMSE"]:.2f}', alpha=0.4, linestyle='--')
    ax6.set_xlabel('Test Örneği', color=TXT)
    ax6.set_ylabel('RUL', color=TXT)
    ax6.legend(facecolor='#111d2e', labelcolor=TXT, fontsize=9, ncol=5)

    fig.suptitle(
        f'NASA C-MAPSS {subset} — Stacking Ensemble v2 (Early Stopping)\n'
        f'Referans RMSE={prev_rmse:.3f} → En iyi={best_new_rmse:.3f}  '
        f'(%{iyilesme:.1f} iyileşme)',
        color=TXT, fontsize=13, fontweight='bold', y=1.01
    )

    OUT = f'stacking_v2_{subset}.png'
    plt.savefig(OUT, dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
    plt.show()
    print(f"✓ Grafik: {OUT}")
    return OUT

In [12]:
# ╔══════════════════════════════════════════════════╗
# ║  HÜCRE 9 — Ana Pipeline (tek subset)            ║
# ╚══════════════════════════════════════════════════╝

# ── Referans değerleri (v1 tablondan) ─────────────
REF = {
    'Bi-LSTM (50ep)':    {'RMSE': 13.430, 'R2': 0.7959, 'Score': 30901.6},
    'LSTM (50ep)':       {'RMSE': 13.489, 'R2': 0.7941, 'Score': 32451.5},
    'Transformer(50ep)': {'RMSE': 14.093, 'R2': 0.7752, 'Score': 49288.7},
    'CNN-LSTM (50ep)':   {'RMSE': 17.487, 'R2': 0.6539, 'Score': 105951.0},
}

# ── Subset seç ────────────────────────────────────
SUBSET = 'FD001'   # ← FD002 / FD003 / FD004 için değiştir
data   = load_and_prepare(SUBSET)
N      = data['n_feat']

print(f"\n{'='*60}")
print(f"  STACKING ENSEMBLE v2 — {SUBSET}")
print(f"  Early Stopping eklenmiş, overfit önlendi")
print(f"{'='*60}")

# ── ADIM 1: Base Modeller ────────────────────────
print("\n[ADIM 1] Base modeller eğitiliyor (max 200 ep, patience=20)...")

bilstm_m, bilstm_te_preds, bilstm_model = train_base_model(
    BiLSTM_v2(N), data,
    max_epochs=200, lr=1e-3, name='Bi-LSTM v2', patience=20
)

transf_m, transf_te_preds, transf_model = train_base_model(
    TransformerRUL(N), data,
    max_epochs=200, lr=5e-4, name='Transformer', patience=20
)

# ── ADIM 2: OOF Tahminleri ────────────────────────
print("\n[ADIM 2] OOF tahminleri (5-fold, early stopping)...")

oof_bilstm = get_oof_predictions(
    BiLSTM_v2, {'n_feat': N},
    data['X_tr'], data['y_tr'],
    n_splits=5, max_epochs=150, lr=1e-3, patience=15,
    name='Bi-LSTM'
)

oof_transf = get_oof_predictions(
    TransformerRUL, {'n_feat': N},
    data['X_tr'], data['y_tr'],
    n_splits=5, max_epochs=150, lr=5e-4, patience=15,
    name='Transformer'
)

# ── ADIM 3: GRU Meta-Model ────────────────────────
print("\n[ADIM 3] GRU Meta-Model eğitiliyor...")

meta_model = GRUMetaModel(n_feat=N, gru_hidden=64)
meta_m, meta_preds = train_meta_model(
    meta_model,
    data['X_tr'], oof_bilstm, oof_transf, data['y_tr'],
    data['X_te'], bilstm_te_preds, transf_te_preds, data['y_te'],
    max_epochs=100, lr=5e-4, patience=20
)

# ── Basit α ensemble ────────────────────────────
best_a, best_simple_rmse = 0.5, float('inf')
for a in np.arange(0, 1.01, 0.05):
    r = float(np.sqrt(mean_squared_error(
        data['y_te'], a*bilstm_te_preds + (1-a)*transf_te_preds
    )))
    if r < best_simple_rmse:
        best_simple_rmse = r; best_a = a
simple_preds = best_a*bilstm_te_preds + (1-best_a)*transf_te_preds
simple_m     = get_metrics(data['y_te'], simple_preds)

# ── Özet tablo ────────────────────────────────────
prev_rmse    = REF['Bi-LSTM (50ep)']['RMSE']
best_new_rmse = min(simple_m['RMSE'], meta_m['RMSE'])
iyilesme     = (prev_rmse - best_new_rmse) / prev_rmse * 100

print(f"\n{'='*72}")
print(f"  {SUBSET} — TAM KARŞILAŞTIRMA TABLOSU")
print(f"{'='*72}")
print(f"  {'Model':<40} {'RMSE':>7} {'R²':>8} {'Score':>12} {'Best Ep':>8}")
print(f"  {'─'*75}")

for lbl, r in REF.items():
    print(f"  {lbl:<40} {r['RMSE']:>7.3f} {r['R2']:>8.4f} {r['Score']:>12.1f}       —")

print(f"  {'─'*75}")

for lbl, m in [
    ('Bi-LSTM v2 (ES) [yeni]',          bilstm_m),
    ('Transformer (ES) [yeni]',         transf_m),
    (f'Basit Ensemble α={best_a:.2f} [yeni]', simple_m),
    ('GRU Stacking Ensemble ★ [yeni]',  meta_m),
]:
    ep_str = str(m.get('best_ep', '—'))
    mk = " ← EN İYİ" if m['RMSE'] == best_new_rmse else ""
    print(f"  {lbl:<40} {m['RMSE']:>7.3f} {m['R2']:>8.4f} "
          f"{m['Score']:>12.1f} {ep_str:>8}{mk}")

print(f"\n  Referans RMSE  : {prev_rmse:.3f}")
print(f"  En iyi yeni    : {best_new_rmse:.3f}  (%{iyilesme:.1f} iyileşme)")
print(f"  En iyi R²      : {max(bilstm_m['R2'], transf_m['R2'], simple_m['R2'], meta_m['R2']):.4f}")

# ── Görselleştirme ────────────────────────────────
plot_results(
    SUBSET, data,
    bilstm_m, transf_m, simple_m, meta_m,
    bilstm_te_preds, transf_te_preds, simple_preds, meta_preds,
    best_a, REF
)

[FD001] Train: (17731, 30, 15) | Test: (10196, 30, 15) | Özellik: 15

  STACKING ENSEMBLE v2 — FD001
  Early Stopping eklenmiş, overfit önlendi

[ADIM 1] Base modeller eğitiliyor (max 200 ep, patience=20)...

──────────────────────────────────────────────────────────
  Bi-LSTM v2 | max 200 epoch | ES patience=20 | Params: 3,788,033
──────────────────────────────────────────────────────────
  Ep    1/200  Loss= 381.660  RMSE= 20.837  R²=0.5087  lr=0.001000  ES=0/20


KeyboardInterrupt: 

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  HÜCRE 10 — Drive'a Kaydet                      ║
# ╚══════════════════════════════════════════════════╝

from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

SAVE = f'/content/drive/MyDrive/cmapss_stacking_v2_{SUBSET}'
os.makedirs(SAVE, exist_ok=True)

out_img = f'stacking_v2_{SUBSET}.png'
shutil.copy(out_img, SAVE)

rows = []
for lbl, r in REF.items():
    rows.append({'Model': lbl, 'Tip': 'Referans',
                 'RMSE': r['RMSE'], 'R2': r['R2'], 'Score': r['Score'],
                 'Best_Epoch': '—'})
for lbl, m in [
    ('Bi-LSTM v2 ES',            bilstm_m),
    ('Transformer ES',           transf_m),
    (f'Basit Ensemble a={best_a:.2f}', simple_m),
    ('GRU Stacking Ensemble',    meta_m),
]:
    rows.append({'Model': lbl, 'Tip': 'v2',
                 'RMSE':       round(m['RMSE'], 4),
                 'R2':         round(m['R2'], 4),
                 'Score':      round(m['Score'], 1),
                 'Best_Epoch': m.get('best_ep', '—')})

df_res = pd.DataFrame(rows)
df_res.to_csv(f'{SAVE}/stacking_v2_sonuclar_{SUBSET}.csv', index=False)

print(f"\n✓ Kaydedildi: {SAVE}")
print(f"\n{df_res.to_string(index=False)}")